# MPPI tracking user guide

Tube MPPI and robust MPPI compare nominal and actual trajectories under one factor-graph objective. This guide builds a scalar integrator, computes feedback, and inspects how these controllers respond to a perturbed initial state.

The CPU controllers follow the [MPPI-Generic controller implementations](https://github.com/ACDSLab/MPPI-Generic/tree/abcd1ab419cb27efa113573932adbde1e0380835/include/mppi/controllers). Graph errors are summed in GTSAM cost units. Match the horizon normalization when comparing to upstream temperatures and thresholds.

GTSAM Copyright 2010–2026, Georgia Tech Research Corporation. See LICENSE.

Authors: Frank Dellaert and Akash Ratheesh

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/planning/doc/MPPITracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import numpy as np
import gtsam

The state and control symbols index ordinary vector values. The objective penalizes terminal position and control effort. All controllers use the same graph. The dynamics callable returns velocity, and the planner applies the time step.

In [3]:
H, dt = 6, 0.2
X = [gtsam.symbol('x', k) for k in range(H+1)]
U = [gtsam.symbol('u', k) for k in range(H)]
def initial(x):
    v = gtsam.Values()
    for key in X:
        v.insert(key, np.array([float(x)]))
    for key in U:
        v.insert(key, np.zeros(1))
    return v
graph = gtsam.NonlinearFactorGraph()
graph.addPriorVector(X[-1], np.zeros(1), gtsam.noiseModel.Unit.Create(1))
for key in U:
    graph.addPriorVector(key, np.zeros(1), gtsam.noiseModel.Isotropic.Sigma(1, 4.))
def dynamics(x, u):
    return u
planner = gtsam.SamplingPlannerVectorVector(X, U, dt)
tracking = gtsam.MPPITracking(np.eye(1), 64, 2.)
nominal, measured = initial(0.5), initial(2.)

DDP optimizes a local quadratic tracking objective and returns one gain per step. Optional lower and upper input limits clip controls on each forward rollout. A control contraction metric provides a second feedback law for control-affine dynamics. For the scalar integrator the identity metric and rate one prescribe a correction equal to minus the state error.

In [4]:
ddp = gtsam.DDPTracking(np.eye(1), np.eye(1), 2.*np.eye(1))
ddp.optimize(np.zeros(1), np.zeros((1, H+1)), np.zeros((1, H)), lambda x,u: x + dt*u, 1,
             lower=np.array([-2.]), upper=np.array([2.]))
def feedback(k, x, ref, u):
    return ddp.gain(k) @ (x-ref)
correction = gtsam.contractionFeedback(np.array([2.]), np.eye(1), np.eye(1), np.zeros(1), 1.)
np.testing.assert_allclose(correction, [-2.])

Tube MPPI evaluates measured and nominal starting states, resets when the measured branch has a sufficiently low baseline, and returns the trajectory obtained by tracking the winning reference. Robust MPPI evaluates intermediate initial states and includes the tracked-system cost in every nominal sample's weight. Its feedback precision sets the cost of the applied correction. For nominal cost $J_n$, actual cost $J_a$, feedback effort $E$, and threshold $T$, the robust nominal score before the likelihood cost is
$$
\tfrac12 J_n + \tfrac12 \max(J_n,\min(J_a+E,T)).
$$
A large deviation therefore has bounded influence on the nominal update, while the actual rollout is weighted by its own cost.

Here both controllers use colored proposals with 64 trajectories per draw. Each controller uses common random perturbations across the branches it compares. Constructing a fresh proposal with the same seed gives each controller a reproducible starting stream. Omitting the proposal uses the planner’s default Gaussian.

In [5]:
def colored_proposal():
    return gtsam.ColoredNoiseProposal(np.eye(1), 64, 2., 42)

tube = tracking.tube(planner, graph, nominal, measured, dynamics, feedback, colored_proposal(),
                    threshold=0., iterations=2)
robust = tracking.robust(planner, graph, nominal, measured, dynamics, feedback, colored_proposal(),
                        threshold=2., candidates=5, stride=1, iterations=2, feedbackPrecision=np.eye(1))
for name, result in [('Tube', tube), ('Robust', robust)]:
    print(name, 'nominal candidate', result.candidate,
          'selected first action', result.actual.atVector(U[0]).item(),
          'actual terminal state', result.actual.atVector(X[-1]).item())
    np.testing.assert_allclose(result.actual.atVector(X[0]), [2.])

Tube nominal candidate 1 selected first action -1.3451967263228506 actual terminal state 0.8673551572265163
Robust nominal candidate 4 selected first action -1.1444816182010418 actual terminal state 0.8627969527110959


In [6]:
from IPython.display import HTML, display
import plotly.graph_objects as go

figure = go.Figure()
for name, solution, color in [("Tube", tube, "#266a74"), ("Robust", robust, "#b75b27")]:
    for path, label, dash in [(solution.nominal, "nominal", "dash"), (solution.actual, "tracked", "solid")]:
        figure.add_scatter(x=np.arange(H + 1) * dt, y=[path.atVector(k).item() for k in X],
                           name=f"{name}, {label}", mode="lines+markers", line=dict(color=color, dash=dash))
figure.update_layout(title="Tracked states over the planning horizon", xaxis_title="Time (s)",
                     yaxis_title="State", template="plotly_white", height=390)
from html import escape

document = figure.to_html(include_plotlyjs="cdn", config={"displaylogo": False, "responsive": True})
display(HTML(f'<iframe title="MPPI tracking user guide" style="width:100%;border:0" '
             f'height="{figure.layout.height or 480}" srcdoc="{escape(document, quote=True)}"></iframe>'))

/Users/dellaert/miniforge3/envs/py312/lib/python3.12/site-packages/IPython/core/display.py:431: UserWarning:

Consider using IPython.display.IFrame instead



Colored MPPI draws temporally correlated noise. Its Tsallis option truncates weights outside the cost cutoff. The primitives controller compares a motion-primitive search with a colored MPPI solution using their minimum sampled costs and a hysteresis margin. The primitive mixture always includes the incumbent and a zero sequence. Additional candidates use colored perturbations, bounded piecewise-linear perturbations around the incumbent, or absolute bounded primitives. A Poisson draw sets the number of switch times separately in each control channel. A stopping sequence can be supplied for a problem-specific high-cost fallback.

In [7]:
colored = tracking.colored(planner, graph, measured, dynamics, 2., 3)
truncated = tracking.colored(planner, graph, measured, dynamics, 2., 3, 2., 2.)
mixture = gtsam.MotionPrimitiveOptions()
mixture.amplitude = np.array([1.5])
mixture.coloredFraction, mixture.shiftedFraction = 0.2, 0.3
mixture.exponent = 2.
primitives = tracking.primitives(planner, graph, measured, measured, dynamics, 2, 2., 0.1, 2, 2, 0., np.zeros((1, H)), mixture=mixture)
for name, plan in [('Colored', colored), ('Tsallis', truncated), ('Primitives', primitives.nominal)]:
    print(name, 'optimized graph error', graph.error(plan))
    assert np.isfinite(graph.error(plan))

Colored optimized graph error 0.4536099072175461
Tsallis optimized graph error 0.5394073794227886
Primitives optimized graph error 0.4797347668859117


Two controls of history enable MPPI-Generic’s five-point quadratic Savitzky–Golay filter at the end of each update. Supply the most recent applied controls, oldest first, at each planning step. A second history can record nominal and actual inputs separately. For scenario costs, `valueAtRisk` interpolates between sorted observations. `conditionalValueAtRisk` averages that quantile and the observations at or above its upper index, following MPPI-Generic’s empirical convention.

In [8]:
tracking.setControlHistory(np.zeros((1, 2)))
smoothed = tracking.colored(planner, graph, measured, dynamics, 2., 2)
assert np.isfinite(graph.error(smoothed))
scenario_costs = np.array([8., 0., 2., 4.])
print("Median scenario cost", gtsam.valueAtRisk(scenario_costs, 0.5))
print("Upper tail cost", gtsam.conditionalValueAtRisk(scenario_costs, 0.5))

Median scenario cost 3.0
Upper tail cost 5.0
